# S5.G2R-2 · A_secure（150 棵，超限补训）：精度与成本

> 按 DR-M5-011 · D-2：A_secure − A 按 S6.P3 口径计算（Holm 按完整次要家族 9 项），但**超出预注册单次拟合上限**，单列为「超限补训」，不改写 S5.G2 的预注册结论。另报 A_secure − L4（同为 150 棵）与 A_secure − L1_k10（描述性）。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
BYTES_PER_GIB = 1024 ** 3
D4_MIN_ABS_DIFF = 0.005          # DR-M5-010 · D-4：「可分辨」但差值小于此值时降级
D1_MAX_SLOWDOWN = 1.5            # DR-M5-011 · D-1：补训段每棵耗时超过 S5.G2 均值的此倍数则补训作废

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g2r_ssxgb_full150.yaml"
rr = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
BASE_PATH = REPO_ROOT / rr["base_config"]
run = yaml.safe_load(BASE_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]
SSXGB_PATH = REPO_ROOT / "modules/m5_modeling/components/g3_ssxgb.py"


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("base config:", BASE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(BASE_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("决策记录:", rr["decision_record"], "| 偏离:", rr["override_note"])
print("seed:", rr["seed"], "| seeds:", rr["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", rr["step_id"], "|", rr["purpose"])
print("SS-XGB 组件:", SSXGB_PATH.relative_to(REPO_ROOT), "sha256:" + sha(SSXGB_PATH))
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = rr["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g2r_ssxgb_full150.yaml sha256:16ec2b9257f2
base config: modules/m5_modeling/configs/s5_g2_ssxgb_run.yaml sha256:0ad22448d9fc
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
决策记录: registry/decision_records/DR-M5-011.yaml | 偏离: owner_override_cap（DR-M5-011）
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 75016e5（工作区含本步未提交改动）
step: S5.G2R | 所有者指示把 A_secure 从 141 棵续训到 150 棵（超出预注册单次拟合上限），按 S6.P3 口径报告 A_secure − A 并标注偏离
SS-XGB 组件: modules/m5_modeling/components/g3_ssxgb.py sha256:0f55ee4a8826
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_ssxgb as SX
from modules.m6_evaluation.components import metrics as MT
from modules.m6_evaluation.components import paired_eval as PE

g1 = yaml.safe_load((REPO_ROOT / "modules/m5_modeling/configs/s5_g1_santander_run.yaml").read_text(encoding="utf-8"))
arr = np.load(PRIV / run["arrays_file"])
local = np.load(PRIV / g1["local_scores_file"])
TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
TOTAL, CAP = TUNING["A"]["best"]["num_boost_round"], pre["compute_caps"]["max_wall_clock_hours_per_fit"]
CKPT = PRIV / rr["ckpt_dir"]
COST = SX.cost_summary(SX.load_progress(str(CKPT), SX.LABEL_PARTY), TOTAL, CAP)
T0 = SX.load_progress(str(PRIV / rr["source_ckpt_dir"]), SX.LABEL_PARTY)["trees_done"]
assert COST["trees"] == TOTAL
y, clusters = arr["dev_eval__y"], arr["dev_eval__cust"]
S = len(rr["seeds"])
SC = {"A_secure": np.load(PRIV / rr["ssxgb_scores_file"]), "A_secure_%d" % T0: np.load(PRIV / run["ssxgb_scores_file"]),
      "A": np.load(PRIV / run["sgb_scores_file"])["dev_eval"]}
SC.update({a: local[a] for a in ("L0", "L1_k10", "L4")})
SC = {a: np.tile(np.asarray(v).reshape(-1), (S, 1)) for a, v in SC.items()}
n_pos, base = int(y.sum()), float(y.mean())
k = run["top_k_share"]
arms = pd.DataFrame([{"arm": a, "auroc": float(np.mean([MT.auc(y, s) for s in m])),
                      "lift_at_k": float(np.mean([MT.lift_at_k(y, s, k) for s in m])),
                      "recall_at_k": float(np.mean([MT.recall_at_k(y, s, k) for s in m]))} for a, m in SC.items()])
arms["precision_at_k"] = arms["lift_at_k"] * base
arms["trees"] = arms["arm"].map({"A_secure": TOTAL, "A_secure_%d" % T0: T0, "A": TOTAL, "L4": TOTAL})
print("开发评估集：%d 位客户，%d 个正例；A_secure 补训到 %d 棵（超出预注册单次拟合上限，%s）"
      % (len(y), n_pos, TOTAL, rr["override_note"]))
print(arms.round(run["round_auc"]).to_string(index=False))

开发评估集：20351 位客户，586 个正例；A_secure 补训到 150 棵（超出预注册单次拟合上限，owner_override_cap（DR-M5-011））
         arm  auroc  lift_at_k  recall_at_k  precision_at_k  trees
    A_secure 0.8675     5.1197       0.5119          0.1474  150.0
A_secure_141 0.8675     5.0856       0.5085          0.1464  141.0
           A 0.8671     5.2392       0.5239          0.1509  150.0
          L0 0.8227     3.5155       0.3515          0.1012    NaN
      L1_k10 0.8244     3.3449       0.3345          0.0963    NaN
          L4 0.8681     5.1368       0.5137          0.1479  150.0


**读法**：A_secure（150 棵）AUROC 0.8675，与 141 棵时相同（0.8675），与 A（隐语 SGB）0.8671、L4（明文集中式）0.8681 相差都在 0.001 以内；前 10% 名单的提升度从 141 棵的 5.09 升到 5.12，A 为 5.24。

In [3]:
rng = np.random.default_rng(rr["seed"])
alpha, power, delta = run["alpha"], run["power"], pre["primary"]["delta_of_interest"]
planned = [("secondary_over_cap", ("A_secure", "A")), ("descriptive", ("A_secure", "L4")),
           ("descriptive", ("A_secure", "L1_k10"))]
out = []
for kind, (a, b) in planned:
    res = PE.paired_bootstrap(y, SC[a], SC[b], clusters, run["n_bootstrap"], alpha, rng)
    gap = PE.detectable_gap(res, n_pos, base, alpha, power, PE.MDE_CONSERVATIVE)
    out.append({"kind": kind, "contrast": "%s − %s" % (a, b), "diff": res["diff"], "ci_low": res["ci_low"],
                "ci_high": res["ci_high"], "p_value": res["p_value"], "detectable_gap": gap,
                "verdict": PE.verdict(res, gap, delta), "estimate_correlation": res["estimate_correlation"]})
con = pd.DataFrame(out)
ref = pd.read_csv(REPO_ROOT / run["reference_contrasts"])     # Holm：S6.P3 完整次要家族 9 项 = S5.G1 的 8 项 + 本项
p_new = con.loc[0, "p_value"]
p_all = sorted(list(ref.loc[ref["kind"] == "secondary", "p_value"]) + [p_new])
m = len(pre["secondary_contrasts"])
assert len(p_all) == m
adj, running = [], 0.0
for rank, p in enumerate(p_all):
    running = max(running, min(1.0, (m - rank) * p))
    adj.append(running)
con["p_holm"] = [max(a for a, p in zip(adj, p_all) if p == p_new), None, None]       # 并列取最保守
small = con.loc[0, "verdict"].startswith("可分辨") and abs(con.loc[0, "diff"]) < D4_MIN_ABS_DIFF
note = "超出预注册单次拟合上限，所有者指示补训（DR-M5-011）"
con["note"] = [note + ("；不能排除运行间波动（DR-M5-010 · D-4）" if small else ""), note + "；描述性", note + "；描述性"]
print(con.drop(columns=["estimate_correlation"]).round(run["round_auc"]).to_string(index=False))

              kind          contrast    diff  ci_low  ci_high  p_value  detectable_gap       verdict  p_holm                               note
secondary_over_cap      A_secure − A  0.0004 -0.0023   0.0031    0.788          0.0060 在预声明尺度上无可分辨差异     1.0     超出预注册单次拟合上限，所有者指示补训（DR-M5-011）
       descriptive     A_secure − L4 -0.0006 -0.0035   0.0022    0.652          0.0067 在预声明尺度上无可分辨差异     NaN 超出预注册单次拟合上限，所有者指示补训（DR-M5-011）；描述性
       descriptive A_secure − L1_k10  0.0431  0.0327   0.0539    0.000          0.0238        可分辨的提升     NaN 超出预注册单次拟合上限，所有者指示补训（DR-M5-011）；描述性


**读法**：A_secure − A = +0.000 [−0.002, 0.003]，p = 0.79，Holm 校正 p = 1.0，可检出差异 0.006，结论为**在预声明尺度上无可分辨差异**——全程秘密分享相对 SGB 没有可见的精度代价（超限补训）。A_secure − L4 = −0.001 [−0.004, 0.002]；A_secure − L1_k10 = +0.043 [0.033, 0.054]，安全档保留相对 L1 的全部增量。开发评估集 586 个正例低于目标 700；SS-XGB 的运行间波动（DR-M5-010 · D-4）未计入区间。

In [4]:
sgb_t = json.loads((PRIV / run["sgb_timing_file"]).read_text(encoding="utf-8"))
prog = SX.load_progress(str(CKPT), SX.LABEL_PARTY)
peak = max(SX.cost_summary(SX.load_progress(str(CKPT), p), TOTAL, CAP)["peak_rss_mb"] for p in SX.PARTIES)
cost = pd.DataFrame([
    {"arm": "A", "mode": "单进程模拟（debug_mode，无网络）", "trees": TOTAL, "fit_seconds": sgb_t["fit_seconds"],
     "seconds_per_tree": sgb_t["fit_seconds"] / TOTAL, "fit_hours": sgb_t["fit_seconds"] / SX.SECONDS_PER_HOUR,
     "lo_gib_per_tree": None, "peak_rss_mb": None, "resume_overhead_seconds": None, "within_cap": True},
    {"arm": "A_secure", "mode": "两进程生产模式，本机回环（分段续训）", "trees": TOTAL, "fit_seconds": COST["fit_seconds"],
     "seconds_per_tree": COST["mean_tree_seconds"], "fit_hours": COST["fit_seconds"] / SX.SECONDS_PER_HOUR,
     "lo_gib_per_tree": COST["lo_tx_bytes"] / TOTAL / BYTES_PER_GIB, "peak_rss_mb": peak,
     "resume_overhead_seconds": COST["resume_overhead_seconds"], "within_cap": COST["within_cap"]}])
print(cost.round(run["round_rate"]).to_string(index=False))
print("每棵树耗时比 A_secure / A：%.1f 倍（运行方式不同，只作同机相对量级）"
      % (cost.loc[1, "seconds_per_tree"] / cost.loc[0, "seconds_per_tree"]))
res_dir = REPO_ROOT / rr["results_dir"]
res_dir.mkdir(parents=True, exist_ok=True)
arms.to_csv(res_dir / "arms_dev_eval.csv", index=False)
con.to_csv(res_dir / "contrasts_dev_eval.csv", index=False)
cost.to_csv(res_dir / "cost_summary.csv", index=False)
pd.DataFrame([{"n_customers": len(y), "n_positives": n_pos, "trees": TOTAL, "trees_within_cap": T0,
               "cap_hours": CAP, "fit_hours": COST["fit_seconds"] / SX.SECONDS_PER_HOUR,
               "within_cap": COST["within_cap"], "override": rr["override_note"]}]).to_csv(res_dir / "run_summary.csv",
                                                                                       index=False)
print("已写出公开汇总：", sorted(p.name for p in res_dir.iterdir()))

     arm                  mode  trees  fit_seconds  seconds_per_tree  fit_hours  lo_gib_per_tree  peak_rss_mb  resume_overhead_seconds  within_cap
       A 单进程模拟（debug_mode，无网络）    150     3324.996            22.167      0.924              NaN          NaN                      NaN        True
A_secure    两进程生产模式，本机回环（分段续训）    150    23171.728           154.478      6.437           32.145     3346.215                 1321.474       False
每棵树耗时比 A_secure / A：7.0 倍（运行方式不同，只作同机相对量级）
已写出公开汇总： ['arms_dev_eval.csv', 'contrasts_dev_eval.csv', 'cost_summary.csv', 'run_summary.csv']


**读法**：150 棵在同一台 2 核机器上累计 6.44 小时，每棵约 154 秒（含当日较慢的 9 棵），约为 A（SGB，单进程）的 7.0 倍；每棵树两方约 32 GiB 通信。两者运行方式不同，只作同机相对量级，不作部署耗时。A_secure 在预注册上限内只能训到 141 棵，补训没有改变精度结论，只是把成本数字补成完整的 150 棵。

## 结论

1. **精度**：训满 150 棵后，A_secure − A = +0.000 [−0.002, 0.003]，无可分辨差异；相对 L1 仍 +0.043。与 141 棵的描述性结果一致——多训 9 棵既没有改变精度，也没有改变结论。
2. **成本**：150 棵约 6.44 小时（超过预注册上限 6 小时），每棵约为 SGB 的 7 倍耗时、两方约 32 GiB 通信。
3. **定性**：S5.G2 的预注册结论（上限内未完成）不变；本步是所有者指示的超限补训，对比按原口径计算并标注偏离（DR-M5-011）。
4. 公开数据上的开发证据，不代表汇丰真实效果；技术侧自评，未经法务复核（W-001）。